In [22]:
from langchain.prompts import PromptTemplate

In [23]:
physics_template = """你是一位非常聪明的物理教授。
你擅长以简洁易懂的方式回答关于物理的问题。
当你不知道某个问题的答案时，你会坦诚承认。

这是一个问题：
{input}"""

physics_prompt = PromptTemplate(
    input_variables=["input"],
    template = physics_template,
)


math_template = """你是一位很棒的数学家。你擅长回答数学问题。
之所以如此出色，是因为你能够将难题分解成各个组成部分，
先回答这些组成部分，然后再将它们整合起来回答更广泛的问题。

这是一个问题：
{input}"""

math_prompt = PromptTemplate(
    input_variables = ["input"],
    template = math_template,
)

In [31]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

def route_decision(input_dict):
    text = input_dict["input"].lower()
    if "物理" in text:
        return "physics"
    elif "数学" in text:
        return "math"
    else:
        return "default"
router_chain = RunnableLambda(route_decision) 

In [32]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model_name="qwen-plus",
    openai_api_key = os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base = "https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.9, 
    max_tokens=5000,
)

In [33]:
destination_chains = {
    "physics": physics_prompt | llm,
    "math": math_prompt | llm,
}
default_chain = PromptTemplate(input_variables=["input"], template="回答：{input}") | llm

In [34]:
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

multi_branch_chain = RunnableBranch(
    (lambda x: x["route"]=="physics", destination_chains["physics"]),
    (lambda x: x["route"]=="math", destination_chains["math"]),
    default_chain
) | {"result": RunnablePassthrough()}

final_chain = (
    {"route": router_chain, "input": RunnablePassthrough()} | multi_branch_chain
)

In [35]:
final_chain.invoke({"input": "如何理解量子纠缠？"})

{'result': AIMessage(content='量子纠缠是一种奇特的量子现象，它描述了两个或多个粒子之间的一种特殊关联。这种关联使得即使在空间上相隔很远的粒子，它们的状态仍然是相互依赖的。理解量子纠缠可以从以下几个方面入手：\n\n1. **基本概念**：  \n   量子纠缠发生在一对或多对粒子通过某种方式相互作用后，它们的量子状态必须依据整个系统来描述，而结果不能单独描述各个粒子的状态。简单来说，一旦两个粒子发生纠缠，测量其中一个粒子的状态会立即影响到另一个粒子的状态，无论它们之间的距离有多远。\n\n2. **贝尔不等式与非局域性**：  \n   纠缠态的存在可以通过实验验证违反贝尔不等式来证明。这表明量子力学中的某些现象无法用经典的局域隐变量理论解释，从而揭示出量子世界中“非局域性”的特性——即信息似乎可以在超距条件下瞬间传递（但并不违反相对论，因为不能传输实际信号）。\n\n3. **纠缠的应用**：  \n   - **量子计算**：利用纠缠可以实现量子比特之间的高效操作，是构建量子计算机的基础之一。\n   - **量子通信**：例如量子密钥分发（QKD），利用纠缠态确保信息传输的安全性。\n   - **量子隐形传态**：借助纠缠可以实现粒子状态的远程传输。\n\n4. **哲学意义**：  \n   量子纠缠挑战了我们对现实和因果关系的传统认知。爱因斯坦曾将此称为“幽灵般的超距作用”，因为他难以接受这种超越经典物理直觉的现象。然而，现代实验已经证实了纠缠的真实性，并成为量子力学的核心特征之一。\n\n总之，量子纠缠不仅是量子力学中最神秘的现象之一，也是推动量子技术发展的重要资源。尽管其本质仍有许多未解之谜，但它为我们打开了一扇通向全新科学领域的窗口。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 393, 'prompt_tokens': 20, 'total_tokens': 413, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}},